# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

## Honest Split

In Week 5, I evaluated the model using a random train/test split. Here I compare that result with a grouped split to reduce information leakage between similar clients.

In [1]:
!git clone https://github.com/JaswanthhKumar22/flyrank-ml-internship.git

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 123, done.
remote: Counting objects: 100% (123/123), done.
remote: Compressing objects: 100% (94/94), done.
remote: Total 123 (delta 39), reused 78 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (123/123), 1.86 MiB | 10.12 MiB/s, done.
Resolving deltas: 100% (39/39), done.


In [2]:
%cd flyrank-ml-internship

/content/flyrank-ml-internship


In [3]:
import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

In [5]:
import pandas as pd

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

df["stale"] = (df["days_since_last_update"] >= 180).astype(int)
df["visible"] = (df["impressions_90d"] >= 500).astype(int)
df["low_ctr"] = (df["ctr"] < 1).astype(int)

df["baseline_score"] = (
    40 * df["stale"] +
    30 * df["visible"] +
    30 * df["low_ctr"]
)

df["high_priority"] = (df["baseline_score"] >= 70).astype(int)

features = [
    "days_since_last_update",
    "impressions_90d",
    "ctr"
]

X = df[features]
y = df["high_priority"]

X_train_random, X_test_random, y_train_random, y_test_random = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

model_random = RandomForestClassifier(
    random_state=42,
    n_estimators=100
)

model_random.fit(X_train_random, y_train_random)

pred_random = model_random.predict(X_test_random)

random_accuracy = accuracy_score(
    y_test_random,
    pred_random
)

groups = df["client_id"]

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups)
)

X_train_group = X.iloc[train_idx]
X_test_group = X.iloc[test_idx]

y_train_group = y.iloc[train_idx]
y_test_group = y.iloc[test_idx]

model_group = RandomForestClassifier(
    random_state=42,
    n_estimators=100
)

model_group.fit(
    X_train_group,
    y_train_group
)

pred_group = model_group.predict(
    X_test_group
)

group_accuracy = accuracy_score(
    y_test_group,
    pred_group
)

comparison = pd.DataFrame({
    "Validation Split": [
        "Random Split (Week 5)",
        "Grouped Split (Week 6)"
    ],
    "Accuracy": [
        round(random_accuracy, 4),
        round(group_accuracy, 4)
    ]
})

comparison

,Validation Split,Accuracy
0,Random Split (Week 5),1.0
1,Grouped Split (Week 6),1.0


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

## Leakage Audit

I reviewed the final feature set used in my Week 5 model to check for potential data leakage.

The model uses only features that are available before making a prediction. Features that may contain future information or reveal the target indirectly were excluded from training.

In [6]:
import pandas as pd

leakage_audit = pd.DataFrame({
    "Feature": [
        "days_since_last_update",
        "impressions_90d",
        "ctr",
        "client_id",
        "content_id",
        "trend_direction",
        "trend_pct"
    ],
    "Used in Model": [
        "Yes",
        "Yes",
        "Yes",
        "No",
        "No",
        "No",
        "No"
    ],
    "Leakage Risk": [
        "Low",
        "Low",
        "Low",
        "Not Used",
        "Not Used",
        "High",
        "High"
    ],
    "Reason": [
        "Available before prediction",
        "Historical metric",
        "Historical metric",
        "Identifier only",
        "Identifier only",
        "May contain future trend information",
        "May contain future trend information"
    ]
})

leakage_audit

,Feature,Used in Model,Leakage Risk,Reason
0,days_since_last_update,Yes,Low,Available before prediction
1,impressions_90d,Yes,Low,Historical metric
2,ctr,Yes,Low,Historical metric
3,client_id,No,Not Used,Identifier only
4,content_id,No,Not Used,Identifier only
5,trend_direction,No,High,May contain future trend information
6,trend_pct,No,High,May contain future trend information


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

## Claim Rewrite

### Original Claim

The Random Forest model accurately predicts high-priority content.

### Revised Claim

The Random Forest model showed good performance on the evaluation dataset and may help prioritize content for review. The observed results are based on the available dataset and should be used for decision-support rather than automated decision-making.

## Self-check

Before you submit, confirm each line honestly:

- [*] Every section above is filled — markdown thinking AND the code that backs it
- [*] The notebook runs top to bottom with no errors (Runtime → Run all)
- [*] No client names, URLs, or private queries anywhere
- [*] My claims use careful words: observed, measured, directional, decision-support
- [*] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.